In [1]:
!pip install transformers torch torchvision pillow

In [2]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests
import torch
import json

model_name = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)
model = Qwen2VLForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).eval()
print("Qwen2 VL cargado correctamente")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Qwen2 VL cargado correctamente


In [3]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 16.2 MB/s eta 0:00:00


In [4]:
from qwen_vl_utils import process_vision_info
import json
import torch
import re

def analizar_prenda(imagen_url, id_prenda):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": imagen_url},
                {"type": "text", "text": """Eres un experto en moda. Analiza la imagen y lista SOLO las prendas principales que se ven claramente, máximo 4.
Los tipos de prenda posibles son: camisa, camiseta, pantalon, vestido, zapatos, corbata, chaqueta, sudadera, jersey, bañador, falda, abrigo, bolso, gorro o lo que consideres oportun

Para cada prenda devuelve SOLO una lista JSON con este formato exacto:
[
    {
        "tipo": "camisa",
        "color": "blanco",
        "estampado": "liso",
        "formalidad": "formal",
        "temporada": ["primavera", "verano"]
    }
]
No inventes prendas que no veas claramente. No repitas prendas. No escribas nada más, solo el JSON."""}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=1024)

    respuesta = processor.batch_decode(output, skip_special_tokens=True)[0]
    texto = respuesta.split("assistant")[-1].strip().replace("```json", "").replace("```", "").strip()

    if not texto.endswith("]"):
        texto = texto[:texto.rfind("}") + 1] + "]"

    try:
        prendas = json.loads(texto)
    except json.JSONDecodeError:
        objetos = re.findall(r'\{[^{}]+\}', texto)
        prendas = [json.loads(o) for o in objetos]

    resultado = []
    for i, prenda in enumerate(prendas):
        prenda["id"] = f"{id_prenda}_{i+1}"
        prenda["imagen_path"] = imagen_url
        resultado.append(prenda)

    return resultado



In [15]:
import json
import os
from pathlib import Path
from google.colab import drive

# Mount Google Drive
#drive.mount('/content/drive',force_remount=True)

CARPETA_ROPA = "/content/drive/MyDrive/ARMARIO (1)"
# Cargar armario existente si ya hay uno
if os.path.exists("armario.json"):
    with open("armario.json", "r", encoding="utf-8") as f:
        armario = json.load(f)
    rutas_existentes = {p["imagen_path"] for p in armario}
    print(f"Armario existente con {len(armario)} prendas")
else:
    armario = []
    rutas_existentes = set()
    print("Armario nuevo")

# Recorrer todas las imágenes de la carpeta
extensiones = {".jpg", ".jpeg", ".png", ".webp"}
# Ensure the directory exists before iterating
imagenes = [str(p) for p in Path(CARPETA_ROPA).iterdir() if p.suffix.lower() in extensiones]

print(f"Imágenes encontradas en carpeta: {len(imagenes)}")

for i, ruta in enumerate(imagenes):
    if ruta in rutas_existentes:
        print(f"⏭ Ya analizada: {os.path.basename(ruta)}")
        continue

    print(f"Analizando {os.path.basename(ruta)}...")
    try:
        resultado = analizar_prenda(ruta, f"prenda_{i+1:03d}")
        for item in resultado:
            item["imagen_path"] = ruta
            armario.append(item)
            print(f"✓ {item['id']} — {item['tipo']} {item['color']}")
    except Exception as e:
        print(f"✗ Error: {e}")

with open("armario.json", "w", encoding="utf-8") as f:
    json.dump(armario, f, indent=2, ensure_ascii=False)

print(f"\nArmario guardado con {len(armario)} prendas")

Armario existente con 0 prendas
Imágenes encontradas en carpeta: 0

Armario guardado con 0 prendas


In [ ]:
import shutil
shutil.copy("armario.json", "/content/drive/MyDrive/ARMARIO (1)/armario.json")
print("Armario guardado en Drive")